# 04 - Fairness Analysis

Computes per-group recall/precision and demographic-parity / equal-opportunity
gaps for each trained classifier. The protected attribute is configured as
`gender` in `config.yaml`.

In [ ]:
from src.utils import get_spark, load_config
from src.fairness_metrics import per_group_metrics, fairness_gaps

cfg = load_config('../config/config.yaml')
spark = get_spark()
protected = cfg['fairness']['protected_attribute']

In [ ]:
for name in ('logistic_regression', 'random_forest', 'gbt'):
    preds = spark.read.parquet(cfg['data']['output_path'] + f'predictions/{name}/')
    metrics = per_group_metrics(preds, protected)
    gaps = fairness_gaps(metrics)
    print(f'\n=== {name} ===')
    for g, m in metrics.items():
        print(f'  {g}: n={m.n} | TPR={m.tpr:.3f} | FPR={m.fpr:.3f} | precision={m.precision:.3f}')
    print(f'  gaps: {gaps}')

### Notes on bias correction

Once the recall gap is identified, the typical mitigations explored in this
project are: per-group threshold tuning (post-processing), class-weight
rebalancing during training, and reweighing the input distribution. Each is
evaluated against the same AUC / AUPR / fairness-gap dashboard above.